# 4장. 사전 학습 자료 만들기

한국어 위키백과 덤프에서 글을 꺼내고, 같은 문장을 빼고, 품질 규칙으로 거르고, 평가 자료가 학습 자료에 섞였는지 확인한 다음, 거르기 전과 후의 자료로 언어 모델을 학습시켜 비교한다.

**준비:** NSMC의 `ratings_train.txt`, `ratings_test.txt`와 한국어 위키백과 덤프의 첫 조각 `kowiki-latest-pages-articles1.xml-p1p82407.bz2`(94MB, https://dumps.wikimedia.org/kowiki/latest/)를 왼쪽 파일 창에 올린다. 코랩에서는 `!pip install tokenizers`를 먼저 실행한다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 25분쯤 걸렸다.

## 4.1 위키백과 덤프에서 글 꺼내기

In [ ]:
import bz2, re, math, time, os, sys, zlib, random, platform, json
from collections import Counter, defaultdict
from xml.etree.ElementTree import iterparse
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
os.environ["RAYON_NUM_THREADS"] = "2"
import tokenizers
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
torch.set_num_threads(2)

DUMP = "kowiki-latest-pages-articles1.xml-p1p82407.bz2"
pages, kinds = [], Counter()
with bz2.open(DUMP, "rb") as f:
    for _, el in iterparse(f, events=("end",)):
        if el.tag.endswith("}page"):
            if el.findtext("{*}ns") != "0":
                kinds["일반 문서가 아님"] += 1
            elif el.find("{*}redirect") is not None:
                kinds["넘겨주기"] += 1
            else:
                kinds["일반 문서"] += 1
                pages.append((el.findtext("{*}title"), el.findtext("{*}revision/{*}text") or ""))
            el.clear()                                  # 읽은 문서는 메모리에서 지운다
print(sum(kinds.values()), dict(kinds), sum(len(t) for _, t in pages))
print(sys.version.split()[0], torch.__version__, tokenizers.__version__, platform.machine(), torch.get_num_threads())

In [ ]:
def strip_nested(text, left, right):
    """가장 안쪽의 left ... right를 지우기를 더 지울 것이 없을 때까지 한다(틀 {{ }}, 표 {| |})."""
    inner = re.compile(re.escape(left) + r"(?:(?!" + re.escape(left) + r").)*?" + re.escape(right), re.S)
    while True:
        new = inner.sub("", text)
        if new == text:
            return new
        text = new

def clean(wiki):
    """위키 문법을 걷어 내고 문단(줄) 목록을 돌려준다."""
    t = re.sub(r"<!--.*?-->", "", wiki, flags=re.S)                       # 주석
    t = re.sub(r"<ref[^>]*/>", "", t)                                      # 각주
    t = re.sub(r"<ref[^>]*>.*?</ref>", "", t, flags=re.S)
    t = strip_nested(t, "{{", "}}")                                        # 틀(정보 상자 등)
    t = strip_nested(t, "{|", "|}")                                        # 표
    t = re.sub(r"<(math|gallery|timeline|score|syntaxhighlight|source|pre|code)[^>]*>.*?</\1>", "", t, flags=re.S | re.I)
    t = re.sub(r"<[^>]+>", "", t)                                          # 남은 HTML 태그
    t = re.sub(r"\[\[(파일|File|그림|Image|분류|Category):[^\[\]]*(\[\[[^\]]*\]\][^\[\]]*)*\]\]", "", t, flags=re.I)
    t = re.sub(r"\[\[([^\]|]*)\|([^\]]*)\]\]", r"\2", t)                   # [[문서|보이는 글]] → 보이는 글
    t = re.sub(r"\[\[([^\]]*)\]\]", r"\1", t)                              # [[문서]] → 문서
    t = re.sub(r"\[https?://[^\s\]]+ ([^\]]*)\]", r"\1", t)                # [주소 글] → 글
    t = re.sub(r"\[https?://[^\]]*\]", "", t)
    t = re.sub(r"'{2,}", "", t)                                            # 굵게·기울임
    t = re.sub(r"__[A-Z]+__", "", t)
    out = []
    for line in t.split("\n"):
        line = line.strip()
        if not line or line.startswith(("=", "|", "!", "{", "}")):          # 제목, 표의 나머지
            continue
        line = re.sub(r"^[*#:;]+\s*", "", line)                            # 목록 기호
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            out.append(line)
    return out

def sentences(paragraph):
    return [s.strip() for s in re.split(r"(?<=[.!?。])\s+", paragraph) if s.strip()]

lap = time.perf_counter()
docs = [(title, clean(text)) for title, text in pages]
sents = [(d, s) for d, (_, lines) in enumerate(docs) for p in lines for s in sentences(p)]
print(f"{time.perf_counter() - lap:.0f}초", sum(len(l) for _, l in docs), len(sents), sum(len(s) for _, s in sents))
show = next(i for i, (t, _) in enumerate(pages) if t == "머리뼈")
print(pages[show][1][:380])
print("→", docs[show][1][:3])

In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import font_manager
for path in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf"))
             + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(path)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 8, "axes.unicode_minus": False})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"

import textwrap
raw_lines = pages[show][1][:520].split("\n")[:12]
fig, axes = plt.subplots(1, 2, figsize=(5.2, 4.8), dpi=260, gridspec_kw={"width_ratios": [1.1, 1]})
for ax, title, lines, color in [(axes[0], "① 덤프의 원문(위키 문법)", raw_lines, RED),
                                (axes[1], "② clean()을 거친 문단", docs[show][1][:3], GREEN)]:
    ax.set_title(title, loc="left", fontsize=8.8, color="#374151")
    y = 0.95
    for line in lines:
        for part in textwrap.wrap(line, 27 if ax is axes[0] else 22) or [""]:
            if y < 0.03:
                break
            ax.text(0.03, y, part, fontsize=8, va="top", color="#B42318" if ax is axes[0] else "#1f2937",
                    transform=ax.transAxes)
            y -= 0.058
        if ax is axes[1]:
            y -= 0.04                                    # 문단 사이를 띄운다
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, fill=False, ec=color, lw=1.2))
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
fig.tight_layout(); fig.savefig("fig_4_clean.png"); plt.close(fig)

## 4.2 같은 문장 빼기: 정확한 중복과 MinHash

In [ ]:
seen, exact = set(), []
for d, s in sents:
    if s not in seen:                                    # 처음 나온 것만 남긴다
        seen.add(s)
        exact.append((d, s))
same = Counter(s for _, s in sents)
print(len(sents), len(exact), len(sents) - len(exact))
print(same.most_common(6))

In [ ]:
P = (1 << 31) - 1                                        # 2^31 − 1, 소수
gen = np.random.default_rng(0)
A_hash = gen.integers(1, P, 128).astype(np.uint64)       # 해시 함수 128개: h_i(x) = (a_i·x + b_i) mod P
B_hash = gen.integers(0, P, 128).astype(np.uint64)

def grams(s, k=5):
    s = re.sub(r"\s+", " ", s)
    return {s[i:i + k] for i in range(max(1, len(s) - k + 1))}

def signature(g):
    """글자 5-그램 집합의 MinHash 서명: 해시 함수마다 가장 작은 값 128개."""
    x = np.fromiter((zlib.crc32(t.encode()) % P for t in g), dtype=np.uint64, count=len(g))
    return ((A_hash[:, None] * x[None, :] + B_hash[:, None]) % P).min(axis=1)

jaccard = lambda a, b: len(a & b) / len(a | b)
long_ones = [(d, s) for d, s in exact if len(s) >= 20]   # 짧은 문장은 정확한 중복만 본다
lap = time.perf_counter()
G = [grams(s) for _, s in long_ones]
SIG = np.array([signature(g) for g in G])
print(len(long_ones), f"{time.perf_counter() - lap:.0f}초")

# 1) 서명이 실제 자카드를 잘 어림하는지: 무작위 쌍과 비슷한 쌍을 섞어 맞춰 본다
pick = random.Random(0)
pairs = [(pick.randrange(len(G)), pick.randrange(len(G))) for _ in range(1500)]
order = sorted(range(len(long_ones)), key=lambda i: long_ones[i][1][:12])  # 앞 12글자가 같은 이웃끼리는 비슷할 수 있다
pairs += [(order[i], order[i + 1]) for i in pick.sample(range(len(order) - 1), 1500)]
check = np.array([(jaccard(G[a], G[b]), (SIG[a] == SIG[b]).mean()) for a, b in pairs if a != b])
print(len(check), f"평균 절대 차이 {np.abs(check[:, 0] - check[:, 1]).mean():.4f}")

In [ ]:
# 2) 서명을 8개씩 16띠로 나눠, 한 띠라도 같으면 후보 쌍으로 본다
buckets = defaultdict(list)
for i in range(len(SIG)):
    for b in range(16):
        buckets[b, SIG[i, b * 8:(b + 1) * 8].tobytes()].append(i)
cand = {(v[a], v[c]) for v in buckets.values() if 1 < len(v) <= 200
        for a in range(len(v)) for c in range(a + 1, len(v))}
near = sorted((a, b) for a, b in cand if jaccard(G[a], G[b]) >= 0.8)
drop = set()
for a, b in near:
    if a not in drop:                                    # 앞 문장이 남아 있으면 뒤 문장을 뺀다
        drop.add(b)
dedup = [x for i, x in enumerate(long_ones) if i not in drop] + [(d, s) for d, s in exact if len(s) < 20]
print(len(cand), len(near), len(drop), len(dedup))
for a, b in random.Random(1).sample(near, 4):
    print(f"{jaccard(G[a], G[b]):.2f} | {long_ones[a][1][:44]} | {long_ones[b][1][:44]}")
s_curve = lambda j: 1 - (1 - j ** 8) ** 16                # 자카드 j인 쌍이 후보가 될 확률
print([round(s_curve(j), 3) for j in (0.5, 0.6, 0.7, 0.8, 0.9)])

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5.2, 5.8), dpi=260, gridspec_kw={"height_ratios": [1.25, 1]})
ax = axes[0]
ax.scatter(check[:, 0], check[:, 1], s=4, color=BLUE, alpha=0.35, lw=0)
ax.plot([0, 1], [0, 1], color=GRAY, lw=1, ls="--")
ax.set_xlabel("실제 자카드 유사도(글자 5-그램 집합)"); ax.set_ylabel("MinHash 어림값\n(서명 128개 중 같은 비율)")
ax.set_title(f"① 쌍 {len(check):,}개: 점선에 가까울수록 어림이 맞다\n(평균 절대 차이 {np.abs(check[:, 0] - check[:, 1]).mean():.3f})",
             loc="left", fontsize=8.6, color="#374151")
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02); ax.grid(color="#eef0f3", lw=0.6)
ax = axes[1]
j = np.linspace(0, 1, 201)
ax.plot(j, s_curve(j), color=ORANGE, lw=2)
ax.axvline(0.8, color=RED, lw=1, ls=":")
ax.text(0.82, 0.42, "0.8 이상을\n같은 문장으로", ha="left", fontsize=8, color=RED)
for jj in (0.5, 0.7, 0.8, 0.9):
    ax.plot([jj], [s_curve(jj)], "o", color=ORANGE, ms=4)
    ax.annotate(f"{s_curve(jj):.3f}", (jj, s_curve(jj)), xytext=(6, -10), textcoords="offset points", fontsize=8, color="#C2560A")
ax.set_xlabel("두 문장의 자카드 유사도"); ax.set_ylabel("후보 쌍이 될 확률")
ax.set_title("② 띠 16개 × 8줄: 비슷한 쌍만 후보로 고른다", loc="left", fontsize=8.6, color="#374151")
ax.grid(color="#eef0f3", lw=0.6)
for a in axes:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_4_minhash.png"); plt.close(fig)

## 4.3 품질 규칙으로 거르기

In [ ]:
hangul = lambda s: sum("가" <= ch <= "힣" for ch in s)
RULES = {
    "짧다(10글자 미만)": lambda s: len(s) < 10,
    "한글이 전체 글자의 50% 미만": lambda s: hangul(s) / len(s) < 0.5,
    "틀을 지운 자국": lambda s: bool(re.search(r"\(\s*[,;]?\s*\)|,\s*,|\(\s*,", s)),
    "끝맺지 않았다": lambda s: not re.search(r"[.!?。」』\"'”)]$", s),
}
caught = {name: [s for _, s in dedup if rule(s)] for name, rule in RULES.items()}
for name, got in caught.items():
    print(f"{name}: {len(got):,}  예: {random.Random(2).sample(got, 3)}")

In [ ]:
# 한글 비율 규칙이 버리는 문장 가운데, 마침표로 끝나는 긴 문장을 골라 본다
suspect = [s for s in caught["한글이 전체 글자의 50% 미만"] if len(s) >= 40 and s.endswith("다.")]
print(len(suspect), random.Random(3).sample(suspect, 4))
# 고친 규칙: 숫자·공백·부호를 빼고 '글자'(isalpha) 가운데 한글의 비율을 본다
RULES["한글이 문자의 50% 미만"] = lambda s: hangul(s) / max(1, sum(ch.isalpha() for ch in s)) < 0.5
del RULES["한글이 전체 글자의 50% 미만"]
caught["한글이 문자의 50% 미만"] = [s for _, s in dedup if RULES["한글이 문자의 50% 미만"](s)]
print("고친 규칙", len(caught["한글이 문자의 50% 미만"]), "고친 뒤에도 걸리는 의심 문장", sum(RULES["한글이 문자의 50% 미만"](s) for s in suspect))
good = [(d, s) for d, s in dedup if not any(rule(s) for rule in RULES.values())]
print(len(good), sum(len(s) for _, s in good))

In [ ]:
steps = [("일반 문서에서 꺼낸 문장", len(sents)), ("정확한 중복을 뺀 뒤", len(exact)),
         ("근사 중복(MinHash)을 뺀 뒤", len(dedup)), ("품질 규칙을 통과한 문장", len(good))]
fig, axes = plt.subplots(2, 1, figsize=(5.2, 5.6), dpi=260, gridspec_kw={"height_ratios": [1, 1.1]})
ax = axes[0]
for i, (name, n) in enumerate(steps):
    ax.barh(i, n, color=[GRAY, BLUE, PURPLE, GREEN][i], height=0.6)
    ax.text(n, i, f"  {n:,}", va="center", fontsize=8.4)
    if i:
        ax.text(n / 2, i, f"−{steps[i - 1][1] - n:,}", va="center", ha="center", fontsize=8, color="white", fontweight="bold")
ax.set_yticks(range(4)); ax.set_yticklabels([n for n, _ in steps], fontsize=8); ax.invert_yaxis()
ax.set_xlim(0, len(sents) * 1.32); ax.set_xlabel("문장 수(만 개)")
ax.set_xticks(range(0, 2_000_001, 500_000))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 10000:,.0f}"))
ax.set_title("① 단계마다 남는 문장(흰 숫자는 그 단계에서 뺀 수)", loc="left", fontsize=8.6, color="#374151")
ax = axes[1]
names = list(caught)
only = {n: sum(1 for s in caught[n] if sum(RULES[m](s) for m in RULES if m != n) == 0) for n in names if n in RULES}
vals = [len(caught[n]) for n in names]
ax.barh(range(len(names)), vals, color=[ORANGE if n not in RULES else BLUE for n in names], height=0.6)
for i, n in enumerate(names):
    tail = f"  {len(caught[n]):,}" + (f" (혼자 {only[n]:,})" if n in only else " (쓰지 않음)")
    ax.text(len(caught[n]), i, tail, va="center", fontsize=8)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8); ax.invert_yaxis()
ax.set_xlim(0, max(vals) * 1.95); ax.set_xlabel("규칙에 걸린 문장 수(만 개, 근사 중복을 뺀 뒤)")
ax.set_xticks(range(0, 500_001, 100_000))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 10000:,.0f}"))
ax.set_title("② 규칙마다 걸린 문장(괄호는 그 규칙에만 걸린 수,\n주황은 좋은 문장까지 버려서 쓰지 않은 처음 규칙)", loc="left", fontsize=8.6, color="#374151")
for a in axes:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_4_funnel.png"); plt.close(fig)

## 4.4 평가 자료가 학습 자료에 섞였는지 확인하기

In [ ]:
tr = pd.read_csv("ratings_train.txt", sep="\t").dropna(subset=["document"])
te = pd.read_csv("ratings_test.txt", sep="\t").dropna(subset=["document"])
norm = lambda t: re.sub(r"[\s\W_]+", "", t)               # 공백과 부호를 뺀다
in_train = te.document.isin(set(tr.document))
in_train_norm = te.document.map(norm).isin(set(map(norm, tr.document)))
print(len(tr), len(te), in_train.sum(), in_train_norm.sum())
print(Counter(te.document[in_train]).most_common(5))
print("위키백과 문장과 같은 시험 문장", te.document.isin({s for _, s in sents}).sum())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3), min_df=2, sublinear_tf=True)
clf = LogisticRegression(max_iter=2000, C=4).fit(vec.fit_transform(tr.document), tr.label)
right = clf.predict(vec.transform(te.document)) == te.label.values
mask = in_train.values
print(f"전체 {right.mean():.4f}  섞인 문장 {right[mask].mean():.4f}({mask.sum():,})  나머지 {right[~mask].mean():.4f}({(~mask).sum():,})")
major = tr.groupby("document").label.agg(lambda x: x.mode()[0])
agree = (te.document[mask].map(major) == te.label[mask]).mean()
print(f"섞인 시험 문장의 레이블이 학습 쪽 레이블과 같은 비율 {agree:.4f}")

In [ ]:
# 위키 문서를 학습용 90%, 검증용 10%로 나눈다(문서 단위)
doc_ids = sorted({d for d, _ in sents})
held = set(random.Random(0).sample(doc_ids, len(doc_ids) // 10))
pool = {"정리만": [s for d, s in sents if d not in held],
        "중복 제거": [s for d, s in dedup if d not in held],
        "중복 제거+규칙": [s for d, s in good if d not in held]}
val_raw = [s for d, s in sents if d in held]
val_good = [s for d, s in good if d in held]
train_docs_sents = set(pool["정리만"])
leak = np.array([s in train_docs_sents for s in val_raw])
print(len(held), [len(v) for v in pool.values()], len(val_raw), len(val_good))
print(f"검증 문장 가운데 학습용 문서에도 똑같이 있는 문장 {leak.mean():.4f}", Counter(np.array(val_raw)[leak]).most_common(5))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5.2, 5.0), dpi=260)
ax = axes[0]
vals = [right[mask].mean() * 100, right[~mask].mean() * 100, right.mean() * 100]
labels = ["학습 파일에도 있는 시험 문장", "학습 파일에 없는 시험 문장", "시험 문장 전체"]
for i, (v, n, c) in enumerate(zip(vals, [mask.sum(), (~mask).sum(), len(mask)], [RED, BLUE, GRAY])):
    ax.plot([86, v], [i, i], color=c, lw=1.2, alpha=0.5)
    ax.plot([v], [i], "o", color=c, ms=7)
    ax.text(v + 0.12, i, f"{v:.2f}%  ({n:,}개)", va="center", fontsize=8)
ax.set_yticks(range(3)); ax.set_yticklabels(labels); ax.set_ylim(2.5, -0.5)
ax.set_xlim(86, 90.5); ax.set_xlabel("감성 분류 정확도(%, 가로축은 86%부터)")
ax.set_title("① NSMC 시험 문장: 학습 파일과 겹친 문장의 정확도", loc="left", fontsize=8.6, color="#374151")
ax = axes[1]
lens = np.array([len(s) for s in val_raw])
for name, sel, color in [("학습용 문서에도 있는 문장", leak, RED), ("없는 문장", ~leak, BLUE)]:
    ax.hist(np.clip(lens[sel], 0, 120), bins=np.arange(0, 125, 5), color=color, alpha=0.6,
            weights=np.full(sel.sum(), 100 / sel.sum()),
            label=f"{name}({sel.sum():,}개, 중앙값 {int(np.median(lens[sel]))}글자)")
ax.set_xlabel("문장 길이(글자, 5글자씩 묶음, 120 넘으면 120에)"); ax.set_ylabel("문장의 비율(%)")
ax.legend(frameon=False, fontsize=8)
ax.set_title("② 위키 검증 문장: 학습용 문서와 똑같은 문장은 짧다", loc="left", fontsize=8.6, color="#374151")
for a in axes:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_4_contam.png"); plt.close(fig)

## 4.5 거른 자료로 학습시키기

In [ ]:
pick = random.Random(0)
train = {"정리만": pick.sample(pool["정리만"], 110000)}
budget = sum(len(s[:64]) for s in train["정리만"])          # 학습 글자 수를 맞춘다
for name in ["중복 제거", "중복 제거+규칙"]:
    shuffled, total, got = pick.sample(pool[name], len(pool[name])), 0, []
    for s in shuffled:
        if total >= budget:
            break
        got.append(s); total += len(s[:64])
    train[name] = got
vals = {"검증(정리만)": pick.sample(val_raw, 10000), "검증(거른 것)": pick.sample(val_good, 10000)}
vals["검증(정리만, 학습에 있는 문장 뺌)"] = pick.sample([s for s, l in zip(val_raw, leak) if not l], 10000)

letters = {ch for ch, n in Counter(ch for s in pick.sample(pool["정리만"], 300000) for ch in s).items() if n >= 5}
UNK_CH, SP = "□", "▁"
to_text = lambda t: "".join(ch if ch in letters else UNK_CH for ch in t[:64]).replace(" ", SP)
tok = Tokenizer(models.BPE(unk_token=UNK_CH))
tok.pre_tokenizer = pre_tokenizers.Split(SP, behavior="merged_with_next")
tok.train_from_iterator([to_text(t) for t in pick.sample(pool["정리만"], 200000)], trainers.BpeTrainer(
    vocab_size=8000, show_progress=False, initial_alphabet=sorted(letters - {" "} | {SP, UNK_CH}),
    special_tokens=["<패딩>", "<시작>", "<끝>"]))
PAD, START, END = 0, 1, 2
SIZE = tok.get_vocab_size()
assert SIZE == 8000 and len(letters) + 2 + 3 < SIZE                # 글자를 다 담고도 병합할 자리가 남아야 한다

def to_tensor(texts):
    rows = [[START] + e.ids + [END] for e in tok.encode_batch([to_text(t) for t in texts])]
    X = torch.zeros(len(rows), max(map(len, rows)), dtype=torch.long)
    for i, r in enumerate(rows):
        X[i, :len(r)] = torch.tensor(r)
    return X

X = {k: to_tensor(v) for k, v in train.items()}
V = {k: (to_tensor(v), sum(len(t[:64]) + 1 for t in v)) for k, v in vals.items()}
for k, v in train.items():
    print(f"{k:<10} 문장 {len(v):>7,}  글자 {sum(len(s[:64]) for s in v):>9,}  토큰 {(X[k] != PAD).sum().item() - 2 * len(v):>9,}")
alphabet = sorted(letters - {" "} | {SP, UNK_CH})
print(len(letters), SIZE - 3 - len(alphabet), len(json.loads(tok.to_str())["model"]["merges"]), f"{np.mean([ch not in letters for s in vals['검증(정리만)'] for ch in s[:64]]):.4f}")

In [ ]:
class SelfAttention(nn.Module):
    """1~3장과 같은 셀프 어텐션(학습하는 위치 표를 쓰는 2장의 모델)."""
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.to_q, self.to_k, self.to_v = (nn.Linear(dim, dim) for _ in range(3))
        self.out = nn.Linear(dim, dim)

    def forward(self, x, hide):
        B, L, D = x.shape
        H = self.heads
        split = lambda t: t.view(B, L, H, D // H).transpose(1, 2)
        q, k, v = split(self.to_q(x)), split(self.to_k(x)), split(self.to_v(x))
        scores = q @ k.transpose(-2, -1) / math.sqrt(D // H)
        scores = scores.masked_fill(hide[:, None, None, :], float("-inf"))
        scores = scores.masked_fill(torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1), float("-inf"))
        return self.out((torch.softmax(scores, dim=-1) @ v).transpose(1, 2).reshape(B, L, D))

class Block(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attend = SelfAttention(dim, heads)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x, hide):
        x = x + self.attend(self.norm1(x), hide)
        return x + self.mlp(self.norm2(x))

class LanguageModel(nn.Module):
    def __init__(self, size=8000, dim=64, heads=4, layers=2, length=66):
        super().__init__()
        self.table = nn.Embedding(size, dim)
        self.where = nn.Embedding(length, dim)
        self.blocks = nn.ModuleList(Block(dim, heads) for _ in range(layers))
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, size)

    def forward(self, ids):
        x = self.table(ids) + self.where(torch.arange(ids.shape[1]))
        hide = ids == PAD
        for block in self.blocks:
            x = block(x, hide)
        return self.head(self.norm(x))

def batches(lengths, seed, size=64, chunk=50):
    shuffler = torch.Generator().manual_seed(seed)
    mixed = torch.randperm(len(lengths), generator=shuffler)
    made = []
    for start in range(0, len(mixed), size * chunk):
        part = mixed[start:start + size * chunk]
        part = part[torch.argsort(lengths[part])]
        made += [part[k:k + size] for k in range(0, len(part), size)]
    return [made[k] for k in torch.randperm(len(made), generator=shuffler)]

def fit(net, X, lr, seed, epochs):
    """1~3장과 같은 학습: AdamW, 배치 64, 처음 10% 걸음 동안 올리고 코사인으로 줄인다."""
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.0)
    lengths = (X != PAD).sum(dim=1)
    total, step = epochs * math.ceil(len(X) / 64), 0
    warm = max(1, total // 10)
    for epoch in range(epochs):
        net.train()
        for rows in batches(lengths, seed * 1000 + epoch):
            step += 1
            for group in optimizer.param_groups:
                group["lr"] = lr * min(1, step / warm) * 0.5 * (1 + math.cos(math.pi * step / total))
            ids = X[rows][:, :int(lengths[rows].max())]
            optimizer.zero_grad()
            out = net(ids[:, :-1])
            F.cross_entropy(out.reshape(-1, out.shape[-1]), ids[:, 1:].reshape(-1), ignore_index=PAD).backward()
            optimizer.step()
    return net

def bpc(net, data, chars):
    net.eval()
    total = 0.0
    with torch.no_grad():
        for k in range(0, len(data), 500):
            ids = data[k:k + 500]
            out = net(ids[:, :-1])
            total += F.cross_entropy(out.reshape(-1, out.shape[-1]), ids[:, 1:].reshape(-1),
                                     ignore_index=PAD, reduction="sum").item()
    return total / math.log(2) / chars

align = lambda text, width: text + " " * max(width - len(text) - sum(ord(c) > 0x1100 for c in text), 1)
print(sum(p.numel() for p in LanguageModel().parameters()))

In [ ]:
results = {}
for name in X:
    for seed in [0, 1, 2]:
        torch.manual_seed(seed)
        lap = time.perf_counter()
        net = fit(LanguageModel(), X[name], 0.01, seed, 4)
        sec = time.perf_counter() - lap
        results[name, seed] = {v: bpc(net, *V[v]) for v in V} | {"sec": sec}
        print(align(name, 14) + f"시드 {seed}  " + "  ".join(f"{results[name, seed][v]:.4f}" for v in V) + f"  {sec:.0f}초", flush=True)

In [ ]:
print(align("학습 자료", 14) + "  ".join(f"{v}" for v in V))
summ = {}
for name in X:
    row = []
    for v in V:
        got = [results[name, s][v] for s in range(3)]
        summ[name, v] = (float(np.median(got)), min(got), max(got))
        row.append(f"{np.median(got):.4f} ({min(got):.4f}~{max(got):.4f})")
    print(align(name, 14) + "  ".join(row))

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.7), dpi=260)
names, vnames = list(X), list(V)
colors = {"정리만": GRAY, "중복 제거": PURPLE, "중복 제거+규칙": GREEN}
for j, v in enumerate(vnames):
    for i, n in enumerate(names):
        m, lo, hi = summ[n, v]
        x = j + (i - 1) * 0.22
        ax.plot([x, x], [lo, hi], color=colors[n], lw=2.6, solid_capstyle="butt")
        ax.plot([x - 0.04, x + 0.04], [lo, lo], color=colors[n], lw=1.2)
        ax.plot([x - 0.04, x + 0.04], [hi, hi], color=colors[n], lw=1.2)
        ax.plot([x], [m], "o", mfc="white", mec=colors[n], mew=1.6, ms=5, label=n if j == 0 else None)
        ax.text(x + 0.065, m, f"{m:.3f}", fontsize=8, va="center", color=colors[n])
ax.set_xticks(range(len(vnames))); ax.set_xticklabels([v.replace("검증(", "검증 문장\n(") for v in vnames], fontsize=8)
ax.set_ylabel("글자당 비트\n(점은 시드 3개의 중앙값, 선은 범위)")
ax.legend(frameon=False, fontsize=8, title="학습 자료", title_fontsize=8, loc="upper left", ncol=3, columnspacing=0.8, handletextpad=0.3)
ax.grid(axis="y", color="#eef0f3", lw=0.6); ax.spines[["top", "right"]].set_visible(False)
lo_all = min(summ[k][1] for k in summ); hi_all = max(summ[k][2] for k in summ)
ax.set_ylim(lo_all - 0.05, hi_all + 0.12)
fig.tight_layout(); fig.savefig("fig_4_results.png"); plt.close(fig)